# nano-vLLM 源码阅读学习计划

> 目标是读懂真实实现，而不是把完整版 vLLM 的能力泛化到本项目。

**适用版本：** 本计划于 2026-10-09 编写，`nano-vllm` 项目的最后提交日期为 2026-04-26。 

## 明确范围：当前实现有哪些，没有哪些

| 主题 | 当前仓库实现 | 学习时的准确表述 |
|---|---|---|
| 模型 | `Qwen3ForCausalLM` | 只精读 Qwen3 路径 |
| 调用形态 | `LLM.generate(prompts, ...)` 一次加入一组请求，内部循环到全部完成 | 离线批量生成，而非持续接收网络请求的服务 |
| 调度 | 维护 `waiting` / `running` 队列；每轮**优先 prefill**，无可 prefill 请求才 decode | 有 iteration-level scheduling，但 prefill 与 decode 不在同一个 `schedule()` 返回批次中混合 |
| chunked prefill | 支持；当 token 预算不足时，只允许队首序列被切块 | 要观察这一具体规则，不笼统称任意请求均可切块 |
| prefix cache | 复用已完成的、按块哈希的 prompt 前缀；使用引用计数 | 复用粒度是完整的 KV block，不是任意 token 前缀 |
| 采样 | temperature + Gumbel-Max；`max_tokens`、`ignore_eos` | 当前没有 top-k、top-p、greedy（`temperature > 1e-10`）或 stop string |
| Attention | FlashAttention 变长 prefill、KV-cache decode；Triton 写缓存 kernel | 应从这些实际接口理解数据流 |
| 性能 | `torch.compile`、CUDA Graph、可选 tensor parallelism | CUDA Graph 仅用于 decode；batch 大于 512 时走 eager 路径 |

阅读过程中请把“项目已经做了什么”和“完整 vLLM 还会做什么”分开写。

## 全局执行图：一次 `generate()` 真实经过的路径

```mermaid
flowchart LR
  A[LLM.generate] --> B[add_request: tokenize → Sequence]
  B --> C[Scheduler.waiting]
  C --> D{Scheduler.schedule}
  D -->|有可调度 prefill| E[prepare_prefill]
  D -->|无 prefill| F[prepare_decode]
  E --> G[ModelRunner.run]
  F --> G
  G --> H[Qwen3ForCausalLM]
  H --> I[Attention: write KV cache + FlashAttention]
  H --> J[ParallelLMHead]
  J --> K[Sampler: temperature / Gumbel-Max]
  K --> L[Scheduler.postprocess]
  L --> C
  L --> M[finished: decode completion tokens]
```

纯文本阅读顺序：LLM.generate → Sequence/Scheduler → ModelRunner → Qwen3 → Attention/KV Cache → Sampler → Scheduler.postprocess → tokenizer.decode。

关键源码锚点：

- `nanovllm/engine/llm_engine.py`：请求加入、主循环和输出整理。
- `nanovllm/engine/scheduler.py`：prefill 优先调度、decode、抢占与完成回收。
- `nanovllm/engine/block_manager.py`：物理块、前缀哈希、引用计数。
- `nanovllm/engine/model_runner.py`：batch 元数据、KV Cache 分配、CUDA Graph。
- `nanovllm/layers/attention.py`：写 KV 与 FlashAttention 调用。
- `nanovllm/models/qwen3.py`：Qwen3 decoder。

# 第一天：跑通并还原主控制流（建议 6–8 小时）

## 第一阶段：最小运行与观察

**目标：** 跑通项目，建立“一个离线批量请求如何完成”的直觉。

1. 阅读 `README.md`、`example.py` 和 `bench.py`。示例使用本地 Qwen3-0.6B 权重、`AutoTokenizer`，并把多条 prompt 一次传给 `generate()`。
2. 按项目说明安装依赖并准备一个本地 Qwen3 模型目录。当前配置断言模型路径必须存在，依赖 CUDA、FlashAttention、Triton 和 PyTorch。
3. 有兼容 NVIDIA GPU 与依赖时，先运行 `example.py` 或等价最小脚本。建议第一次使用 `enforce_eager=True`：这样不捕获 CUDA Graph，调试路径更直接。
4. 没有兼容 GPU 时，不必强行运行：静态追踪控制流，并把基于 GPU 的性能实验标为待补；本项目的 ModelRunner 初始化会调用 CUDA / NCCL。
5. 记录：模型、GPU、`tensor_parallel_size`、`max_model_len`、prompt 数量、每条 prompt token 长度、`temperature` 和 `max_tokens`。

**观察点：** `generate()` 的进度条分别显示 Prefill 与 Decode 吞吐。它们由 `LLMEngine.generate()` 在每轮根据 `step()` 的返回值更新。

**建议时间：** 约 2–3 小时；环境或模型准备受阻时，及时切换到源码阅读。

## 第二阶段：从 `generate()` 追到首个生成 token

按下列顺序阅读并做调用图：

1. `LLM` 只是 `LLMEngine` 的别名式子类；从 `LLMEngine.generate()` 开始。
2. `add_request()`：字符串 prompt 先被 tokenizer 编码，然后创建 `Sequence` 并进入 `Scheduler.waiting`。
3. `step()`：依次执行 `schedule()`、`model_runner.call("run", ...)`、`postprocess()`。
4. `Sequence`：记录 `token_ids`、`num_cached_tokens`、`num_scheduled_tokens`、`block_table` 和状态。
5. 第一次 prefill 完成后，`postprocess()` 才会调用 `append_token(token_id)`；这个 token 是下一轮 decode 的输入。

建议搜索：

```powershell
rg -n "def generate|def step|def schedule|def postprocess|append_token" nanovllm
```

**写作任务：** 用自己的话说明：模型为何在处理 prompt 后已经能采样出一个 token？这个 token 在何时成为下一次 decode 的 `last_token`？

## 第三阶段：理解当前 Scheduler，而不是泛化的 Scheduler

`Scheduler.schedule()` 的逻辑有明确优先级：

1. 只要 `waiting` 中有可调度的 prefill，就组一个 prefill 批次并立即返回。
2. 这个批次同时受 `max_num_seqs` 和 `max_num_batched_tokens` 限制。
3. 若当前 token 预算装不下完整 prompt，只有**队首序列**允许 chunked prefill；后续序列会留在 waiting。
4. 仅当本轮没有任何 prefill 序列可调度时，才从 `running` 取序列做 decode；每条序列此时只调度 1 个 token。
5. decode 若无块可追加，调度器会 preempt 一个运行中序列：释放其块、设回 waiting，之后从头部重新 prefill。

用两个不同长度 prompt 和一个很小的 `max_num_batched_tokens`，在纸上列出每轮的 waiting、running、scheduled、`num_cached_tokens`。

**第一天验收：** 能准确说明“prefill 与 decode 在这个实现中不会混入同一次 `ModelRunner.run()`”。

**建议时间：** 主流程追读约 3 小时，状态机纸上模拟约 1–2 小时。

# 第二天：Paged KV Cache、前缀复用与 Attention（建议 6–8 小时）

## 第一阶段：BlockManager 的真实数据结构

**目标：** 分清 CPU 端的逻辑块管理和 GPU 端的实际 KV 张量。

`BlockManager` 维护：

- `blocks`：包含 `block_id`、`ref_count`、hash 和 token ID 的 Python 元数据；
- `free_block_ids` 与 `used_block_ids`：可用 / 已使用物理块；
- `hash_to_block_id`：完整前缀块哈希到物理块的映射；
- 每个 `Sequence.block_table`：逻辑块序号到物理块 ID 的映射。

`ModelRunner.allocate_kv_cache()` 在 GPU 上实际创建：

```text
[2, num_layers, num_kvcache_blocks, block_size,
 num_kv_heads_per_rank, head_dim]
```

第一维 2 分别对应 key 和 value。默认 `kvcache_block_size = 256`，并且配置断言其为 256 的倍数。

**源码练习：** 对 `allocate()`、`may_append()`、`deallocate()`、`hash_blocks()` 各写一句“改变了什么状态”。注意：最后释放的块若 ref_count 归零会回到 free 队列，但 hash/token 元数据会保留到该物理块下一次被分配时再清除；这是实现 prefix cache 的方式之一。

## 第二阶段：Prefix caching 的精确机制

当前实现的 prefix cache 不是对任意共享文本做模糊匹配：

1. `can_allocate()` 从第一个逻辑块开始计算滚动 xxHash：当前块 token ID 与前一个块的 hash 共同决定新 hash。
2. `can_allocate()` 只检查 `seq.num_blocks - 1` 个逻辑块：最后一个逻辑块完全不参与命中检查，即使它刚好是完整块；不满块的尾部当然也不会被复用。也就是说，复用前缀要在最后一个逻辑块之前结束。
3. 查到 hash 后还会对比 `token_ids`，避免只依赖 hash。
4. `allocate()` 将已命中的物理块放进新序列的 `block_table`，并通过 `ref_count` 共享。
5. `postprocess()` 调用 `hash_blocks()`，只有刚刚变成完整块的部分才被登记到 `hash_to_block_id`。
6. `deallocate()` 递减引用计数，计数归零才归还物理块。

因此最合适的实验不是“两个相似 prompt”，而是两条有多个**相同完整 256-token 块**的 token ID 序列，并在其后继续不同 token，使共同块不落在较短序列的最后一块。实验前先确认模型 tokenization，避免用字符数代替 token 数。

**思考：** 共享块被一个请求释放时，为什么另一个请求仍能继续使用？

## 第三阶段：从 batch 元数据到 FlashAttention

按 `ModelRunner.prepare_prefill()` 和 `prepare_decode()` 阅读数据：

### Prefill

- 输入是从 `num_cached_tokens` 开始的本轮 token；
- 构造 `cu_seqlens_q`、`cu_seqlens_k`、`max_seqlen_q`、`max_seqlen_k`；
- `slot_mapping` 将本轮 K/V 写到物理块中的线性槽位；
- 如果 `cu_seqlens_k[-1] > cu_seqlens_q[-1]`，意味着使用了缓存前缀，于是另传 `block_tables`。

### Decode

- 每个序列只输入 `last_token`；
- `positions = len(seq) - 1`，`context_lens = len(seq)`；
- 为每条序列建立 block table，并定位新 K/V 的写入 slot。

`Attention.forward()` 先由 Triton 的 `store_kvcache` 写入 K/V；之后 prefill 调 `flash_attn_varlen_func`，decode 调 `flash_attn_with_kvcache`。

**第二天验收：** 画一条 token 从 `slot_mapping` 写入物理 KV 缓存，再经 block table 被后续 attention 读取的路径。

**建议时间：** BlockManager 与 prefix cache 约 3–4 小时，Attention 数据流与实验设计约 2–3 小时。

# 第三天：模型、采样、CUDA Graph 与并行（建议 6–8 小时）

## 第一阶段：Qwen3 前向与张量并行

从 `nanovllm/models/qwen3.py` 阅读：

1. `Qwen3ForCausalLM` 由 `Qwen3Model` 与 `ParallelLMHead` 组成。
2. 每层执行 RMSNorm → self-attention → RMSNorm → SwiGLU MLP，并以 residual 流串联。
3. Attention 将 QKV 投影拆成 Q、K、V，reshape 为 head 维度；若配置不使用 QKV bias，会对 Q 与 K 做 RMSNorm；随后应用 RoPE。
4. `num_key_value_heads` 可以小于 `num_attention_heads`，所以 KV Cache 的 head 数按 KV heads 计算。
5. `tensor_parallel_size > 1` 时，`ModelRunner` 用 NCCL 初始化多个 rank，rank 0 通过 shared memory + event 广播方法调用；线性层使用 column / row parallel 分片，row-parallel 输出经 all-reduce 汇总。

学习顺序建议：先看单卡（TP=1）前向，再读 `layers/linear.py` 理解分片和 `all_reduce`。

## 第二阶段：采样和停止条件——以当前代码为准

`SamplingParams` 仅有：

| 参数 | 当前作用 |
|---|---|
| `temperature` | logits 除以 temperature 后 softmax；必须大于 `1e-10` |
| `max_tokens` | completion token 数达到上限即结束 |
| `ignore_eos` | 为 false 时，采样到 tokenizer 的 EOS 即结束 |

`Sampler.forward()` 使用 Gumbel-Max 形式的随机采样。当前项目没有 top-k、top-p、repetition penalty、beam search、stop string，也不允许 `temperature=0` 的 greedy 采样。

**实验：** 固定 prompt 和 max_tokens，分别用 temperature 0.2、0.6、1.0，重复多次并观察输出差异。不要把不存在的 top-p/top-k 参数加入实验。

## 第三阶段：CUDA Graph 与性能实验

初始化 `ModelRunner` 时：

- 先 `warmup_model()`，以配置允许的 token 预算做一次 prefill；
- 再按 `gpu_memory_utilization`、模型层数、KV heads、head dim、dtype 计算可分配的缓存块数；
- `enforce_eager=False` 时捕获 CUDA Graph。

Graph 的使用边界：

- **仅 decode** 尝试 graph replay；
- prefill、`enforce_eager=True` 或实际 decode batch size 大于 512 时走普通 eager 前向；
- graph batch size 取 `[1, 2, 4, 8]` 加 16 的倍数，实际 batch 向上选最近的已捕获大小；
- replay 前，代码拷贝 input IDs、positions、slot mapping、context lengths 和 block tables 到预分配张量。

建议按以下顺序做实验，每次只改一个变量：

1. `enforce_eager=True/False`，比较相同 decode 工作负载；
2. 共享完整 256-token 前缀与不共享前缀，观察缓存复用路径；
3. 修改 `max_num_batched_tokens`，观察 chunked prefill；
4. `tensor_parallel_size=1` 起步，确认多 GPU 环境和 NCCL 正常后再试 TP。

记录 wall time、输入 / 输出 token 数、Prefill / Decode 吞吐、峰值显存和精确配置。没有 GPU 时，沿 eager / graph 分支静态比较，并写明哪些性能结论尚未实测。

**建议时间：** 模型与采样约 3 小时，Graph / TP 机制及实验约 3–5 小时。

# 三天验收与推荐阅读顺序

## 验收清单

- [ ] 能从 `LLMEngine.generate()` 追到 `Sampler`，说明首 token 与后续 decode 的区别。
- [ ] 能手工模拟 waiting / running 和 prefill 优先的调度结果。
- [ ] 能解释块表、slot mapping、GPU KV 张量三者的关系。
- [ ] 能准确说明 prefix cache 只复用可命中的完整前置块（跳过最后一个逻辑块），并用引用计数保护共享块。
- [ ] 能指出当前实现不支持的 sampling 功能。
- [ ] 能说明 CUDA Graph 为何只用于特定 decode 分支。
- [ ] 能从 Qwen3 的 QKV、RoPE、GQA/MQA 与 KV Cache 形状建立联系。

## 阅读顺序

1. `README.md`、`example.py`、`bench.py`
2. `engine/llm_engine.py`
3. `engine/sequence.py`
4. `engine/scheduler.py`
5. `engine/block_manager.py`
6. `engine/model_runner.py`
7. `utils/context.py` 与 `layers/attention.py`
8. `layers/sampler.py`
9. `models/qwen3.py`、`layers/rotary_embedding.py`
10. `layers/linear.py` 与 `utils/loader.py`

完成后再读完整 vLLM 时，优先比较调度策略、block 管理、attention backend、模型注册和在线服务层分别扩展了什么，而不要假设 nano-vLLM 已经完整覆盖它们。